# Slopless Star History

This notebook reads `slopless-stars.csv`, builds a minute-level time series, and plots cumulative stars plus per-minute changes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

csv_path = Path("slopless-stars.csv")
if not csv_path.exists():
    csv_path = Path("analytics/star-history/slopless-stars.csv")
stars = pd.read_csv(csv_path, parse_dates=["starred_at"])
stars = stars.sort_values("starred_at").reset_index(drop=True)
stars["star_number"] = range(1, len(stars) + 1)
stars["minute"] = stars["starred_at"].dt.floor("min")
stars["previous_starred_at"] = stars["starred_at"].shift(1)
stars["seconds_since_previous"] = (stars["starred_at"] - stars["previous_starred_at"]).dt.total_seconds()
stars.head()

In [ ]:
summary = {
    "stars": len(stars),
    "first_star": stars["starred_at"].min(),
    "last_star": stars["starred_at"].max(),
    "active_minutes": stars["minute"].nunique(),
    "max_stars_in_one_minute": stars.groupby("minute").size().max(),
}
summary

In [ ]:
minute_counts = stars.set_index("starred_at").resample("min").size().rename("stars_added")
minute_curve = minute_counts.to_frame()
minute_curve["cumulative_stars"] = minute_curve["stars_added"].cumsum()
minute_curve["stars_added_5min"] = minute_curve["stars_added"].rolling(5, min_periods=1).sum()
minute_curve["stars_added_15min"] = minute_curve["stars_added"].rolling(15, min_periods=1).sum()
minute_curve.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
minute_curve["cumulative_stars"].plot(ax=ax, linewidth=2)
ax.set_title("Cumulative GitHub Stars")
ax.set_xlabel("Time UTC")
ax.set_ylabel("Stars")
ax.grid(True, alpha=0.25)
plt.show()

In [ ]:
active_minutes = minute_curve[minute_curve["stars_added"] > 0]

fig, ax = plt.subplots(figsize=(14, 5))
active_minutes["stars_added"].plot(ax=ax, kind="bar", width=1)
ax.set_title("Stars Added Per Active Minute")
ax.set_xlabel("Active minute UTC")
ax.set_ylabel("Stars added")
ax.tick_params(axis="x", labelbottom=False)
ax.grid(True, axis="y", alpha=0.25)
plt.show()

active_minutes.sort_values("stars_added", ascending=False).head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
stars.plot(x="starred_at", y="seconds_since_previous", ax=ax, style=".", legend=False)
ax.set_title("Seconds Between Consecutive Stars")
ax.set_xlabel("Time UTC")
ax.set_ylabel("Seconds since previous star")
ax.grid(True, alpha=0.25)
plt.show()

stars[["starred_at", "user", "seconds_since_previous"]].tail(30)

In [ ]:
minute_curve.to_csv("slopless-stars-by-minute.csv")
stars.to_csv("slopless-stars-expanded.csv", index=False)
print("wrote slopless-stars-by-minute.csv and slopless-stars-expanded.csv")